# 11. Stacking Scorers: закрывает ли ансамбль разрыв с AutoML?

**Контекст:** LightAutoML (0.815) и AutoGluon (0.806) обходят AutoIntent (~0.76-0.79) на тех же e5-фичах.

**Гипотеза:** Разрыв — за счёт стекинга/ансамбля, а не самих фич.

**ВАЖНО (v4):**
- Стекинг проверяется на **3 сидах** (42, 123, 456), как и AutoML
- **Детерминизм гарантирован**: `thread_count=1` для CatBoost, `n_jobs=1` для RF и CV
- **ROC-кривые** для визуального сравнения всех прогонов

**Методология:**
1. Выяснить РЕАЛЬНЫЙ состав скореров в 4 пресетах (0.2.0 vs 0.3.0, light vs medium)
2. Для каждого из 3 сидов: обучить стекинг на **полном train 100k**
3. Усреднить результаты (mean ± std) для корректного сравнения с AutoML
4. Построить ROC-кривые для всех моделей
5. Сравнить с внешними AutoML и одиночными скорерами **в равных условиях**

In [13]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)

# CatBoost — проверим доступность
try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("⚠️ CatBoost не установлен")

warnings.filterwarnings('ignore')

# === CONFIG ===
BASE = Path("..").resolve()
EMB_CACHE = BASE / "data/processed/embeddings_cache"
DATA_DIR = BASE / "data/processed"
OUTPUT_DIR = BASE / "results/diagnostics"

# 3 сида — как у AutoML
SEEDS = [42, 123, 456]
N_FOLDS = 5

# Eval — один и тот же для всех сидов
EVAL_JSONL = DATA_DIR / "wildjailbreak_eval_binary.jsonl"
EVAL_EMB = EMB_CACHE / "intfloat_multilingual-e5-large-instruct_test.npy"

print(f"Seeds: {SEEDS}")
print(f"Eval embeddings: {EVAL_EMB}")
print(f"CatBoost available: {CATBOOST_AVAILABLE}")

Seeds: [42, 123, 456]
Eval embeddings: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/data/processed/embeddings_cache/intfloat_multilingual-e5-large-instruct_test.npy
CatBoost available: True


## ЭТАП 0: Реальный состав скореров в пресетах

Данные получены из:
- **0.2.0**: локальные YAML файлы `.venv/.../autointent/_presets/`
- **0.3.0**: GitHub dev branch `raw.githubusercontent.com/DeepPavlov/AutoIntent/dev/src/autointent/_presets/`

In [14]:
# === СОСТАВ СКОРЕРОВ (верифицировано из YAML) ===

# autointent 0.2.0 (локальная версия)
scorers_020_light = ["knn", "linear", "mlknn"]
scorers_020_medium = ["knn", "linear", "mlknn", "catboost", "sklearn:RandomForestClassifier"]

# autointent 0.3.0 (GitHub dev branch)
scorers_030_light = ["knn", "linear", "mlknn"]
scorers_030_medium = ["knn", "linear", "mlknn", "sklearn:RandomForestClassifier"]  # БЕЗ catboost!

print("="*80)
print("ЭТАП 0: Реальный состав скореров в пресетах")
print("="*80)

print(f"\n=== autointent 0.2.0 ===")
print(f"classic-light:  {scorers_020_light}")
print(f"classic-medium: {scorers_020_medium}")

print(f"\n=== autointent 0.3.0 (dev branch) ===")
print(f"classic-light:  {scorers_030_light}")
print(f"classic-medium: {scorers_030_medium}")

# Разница
diff_medium = set(scorers_020_medium) - set(scorers_030_medium)
print(f"\n=== РАЗНИЦА 0.2.0 vs 0.3.0 ===")
print(f"classic-light:  без изменений")
print(f"classic-medium: удалено в 0.3.0: {list(diff_medium)}")

ЭТАП 0: Реальный состав скореров в пресетах

=== autointent 0.2.0 ===
classic-light:  ['knn', 'linear', 'mlknn']
classic-medium: ['knn', 'linear', 'mlknn', 'catboost', 'sklearn:RandomForestClassifier']

=== autointent 0.3.0 (dev branch) ===
classic-light:  ['knn', 'linear', 'mlknn']
classic-medium: ['knn', 'linear', 'mlknn', 'sklearn:RandomForestClassifier']

=== РАЗНИЦА 0.2.0 vs 0.3.0 ===
classic-light:  без изменений
classic-medium: удалено в 0.3.0: ['catboost']


## ЭТАП 1: Вспомогательные функции

**Фикс недетерминизма (v4):**
- CatBoost: `thread_count=1`, `task_type="CPU"` — многопоточность вносит рандом из-за разного порядка суммирования
- RandomForest: `n_jobs=1` — аналогично
- cross_val_predict: `n_jobs=1` — для согласованного порядка фолдов

Теперь результаты полностью воспроизводимы при одинаковом seed.

In [15]:
def get_train_paths(seed: int):
    """Пути к train данным для конкретного сида."""
    emb_path = EMB_CACHE / f"intfloat_multilingual-e5-large-instruct_full100k_seed{seed}.npy"
    json_path = DATA_DIR / f"wildjailbreak_full100k_seed{seed}.json"
    return emb_path, json_path


def load_train_data(seed: int):
    """Загрузка train эмбеддингов и меток для сида."""
    emb_path, json_path = get_train_paths(seed)
    
    # Эмбеддинги
    train_emb = np.load(emb_path)
    
    # Метки из JSON
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    n_safe = len(data['intents'][0]['utterances'])
    n_jailbreak = len(data['oos_utterances'])
    train_labels = np.array([0] * n_safe + [1] * n_jailbreak)
    
    return train_emb, train_labels


def load_eval_data():
    """Загрузка eval эмбеддингов и меток (одинаковые для всех сидов)."""
    eval_emb = np.load(EVAL_EMB)
    
    eval_labels = []
    with open(EVAL_JSONL, 'r', encoding='utf-8') as f:
        for line in f:
            eval_labels.append(json.loads(line)['label'])
    eval_labels = np.array(eval_labels)
    
    return eval_emb, eval_labels


def compute_metrics(y_true, y_prob, threshold=0.5):
    """Compute all metrics. Порог 0.5 — стандартный."""
    y_pred = (y_prob >= threshold).astype(int)
    safe_mask = y_true == 0
    orr = y_pred[safe_mask].mean() if safe_mask.sum() > 0 else None
    
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "over_refusal_rate": orr,
    }


def soft_voting(probs_list):
    """Average probabilities."""
    return np.mean(probs_list, axis=0)


def stacking_predict(oof_matrix, y_train, eval_matrix, seed):
    """Train meta-model on OOF and predict on eval."""
    meta = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=seed)
    meta.fit(oof_matrix, y_train)
    return meta.predict_proba(eval_matrix)[:, 1]


def get_base_models(seed: int):
    """Базовые модели с фиксированным сидом. Детерминизм гарантирован."""
    models = {
        "linear": LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=seed
        ),
        "knn": KNeighborsClassifier(
            n_neighbors=10, metric='cosine', weights='distance'
        ),
        "randomforest": RandomForestClassifier(
            n_estimators=150, max_depth=20, min_samples_leaf=5,
            class_weight='balanced', random_state=seed, n_jobs=1  # n_jobs=1 для детерминизма
        ),
    }
    
    if CATBOOST_AVAILABLE:
        models["catboost"] = CatBoostClassifier(
            iterations=300, learning_rate=0.1, depth=6,
            random_seed=seed, verbose=False,
            thread_count=1,  # ФИКС: детерминизм (многопоточность вносит рандом)
            task_type="CPU",  # ФИКС: явно CPU для воспроизводимости
        )
    else:
        raise ImportError("CatBoost ДОЛЖЕН быть установлен!")
    
    return models


# Наборы скореров
PRESETS = {
    "0.2.0/classic-light": ["knn", "linear"],
    "0.2.0/classic-medium": ["knn", "linear", "catboost", "randomforest"],
    "0.3.0/classic-light": ["knn", "linear"],
    "0.3.0/classic-medium": ["knn", "linear", "randomforest"],
}

print("✅ Вспомогательные функции определены")
print(f"\nНаборы моделей для стекинга:")
for preset, models in PRESETS.items():
    print(f"  {preset}: {models}")
print(f"\n⚠️ Детерминизм: thread_count=1 для CatBoost, n_jobs=1 для RF")

✅ Вспомогательные функции определены

Наборы моделей для стекинга:
  0.2.0/classic-light: ['knn', 'linear']
  0.2.0/classic-medium: ['knn', 'linear', 'catboost', 'randomforest']
  0.3.0/classic-light: ['knn', 'linear']
  0.3.0/classic-medium: ['knn', 'linear', 'randomforest']

⚠️ Детерминизм: thread_count=1 для CatBoost, n_jobs=1 для RF


## ЭТАП 2: Обучение стекинга на 3 сидах

In [16]:
# Загрузка eval (одинаковый для всех)
eval_emb, eval_labels = load_eval_data()
print(f"Eval: {eval_emb.shape[0]} samples")

# Проверка наличия данных для всех сидов
print(f"\nПроверка наличия данных:")
for seed in SEEDS:
    emb_path, json_path = get_train_paths(seed)
    emb_ok = emb_path.exists()
    json_ok = json_path.exists()
    status = "✅" if (emb_ok and json_ok) else "❌"
    print(f"  seed={seed}: {status} (emb={emb_ok}, json={json_ok})")

Eval: 2210 samples

Проверка наличия данных:
  seed=42: ✅ (emb=True, json=True)
  seed=123: ✅ (emb=True, json=True)
  seed=456: ✅ (emb=True, json=True)


In [ ]:
# === ГЛАВНЫЙ ЦИКЛ: обучение на 3 сидах ===
all_results = []  # Результаты по всем сидам
all_predictions = {}  # y_prob для ROC кривых: {(seed, preset, method): y_prob}

print("="*80)
print("ОБУЧЕНИЕ СТЕКИНГА НА 3 СИДАХ (как AutoML)")
print("="*80)

for seed in SEEDS:
    print(f"\n{'#'*80}")
    print(f"# SEED = {seed}")
    print(f"{'#'*80}")
    
    # Загрузка train для этого сида
    train_emb, train_labels = load_train_data(seed)
    print(f"\nTrain: {train_emb.shape[0]} samples (safe={sum(train_labels==0)}, jailbreak={sum(train_labels==1)})")
    
    # CV split для этого сида
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    
    # Базовые модели для этого сида
    base_models = get_base_models(seed)
    
    # OOF и eval predictions для базовых моделей
    oof_probs = {}
    eval_probs = {}
    
    print(f"\nОбучение базовых моделей...")
    for name, model in base_models.items():
        print(f"  → {name}...", end=" ")
        
        # OOF
        oof = cross_val_predict(
            model, train_emb, train_labels,
            cv=cv, method='predict_proba', n_jobs=1
        )[:, 1]
        oof_probs[name] = oof
        
        # Full train → eval
        model_full = model.__class__(**model.get_params())
        model_full.fit(train_emb, train_labels)
        eval_probs[name] = model_full.predict_proba(eval_emb)[:, 1]
        
        # Сохраняем базовые модели для ROC
        all_predictions[(seed, f"base:{name}", "single")] = eval_probs[name]
        
        print(f"OOF AUC: {roc_auc_score(train_labels, oof):.4f}, Eval AUC: {roc_auc_score(eval_labels, eval_probs[name]):.4f}")
    
    # Проверка выровненности OOF
    lengths = {name: len(oof) for name, oof in oof_probs.items()}
    assert len(set(lengths.values())) == 1, f"OOF misalignment for seed={seed}"
    
    # Стекинг для каждого набора
    print(f"\nСтекинг по наборам:")
    for preset, model_names in PRESETS.items():
        oof_list = [oof_probs[m] for m in model_names]
        eval_list = [eval_probs[m] for m in model_names]
        
        # Soft-voting
        soft_eval = soft_voting(eval_list)
        soft_m = compute_metrics(eval_labels, soft_eval)
        all_results.append({"seed": seed, "preset": preset, "method": "soft-voting", **soft_m})
        all_predictions[(seed, preset, "soft-voting")] = soft_eval
        
        # Stacking
        oof_matrix = np.column_stack(oof_list)
        eval_matrix = np.column_stack(eval_list)
        stack_eval = stacking_predict(oof_matrix, train_labels, eval_matrix, seed)
        stack_m = compute_metrics(eval_labels, stack_eval)
        all_results.append({"seed": seed, "preset": preset, "method": "stacking", **stack_m})
        all_predictions[(seed, preset, "stacking")] = stack_eval
        
        print(f"  {preset}: soft={soft_m['roc_auc']:.4f}, stack={stack_m['roc_auc']:.4f}")

print(f"\n✅ Обучение завершено для всех {len(SEEDS)} сидов")
print(f"   Сохранено predictions для {len(all_predictions)} прогонов (для ROC кривых)")

ОБУЧЕНИЕ СТЕКИНГА НА 3 СИДАХ (как AutoML)

################################################################################
# SEED = 42
################################################################################

Train: 100000 samples (safe=50000, jailbreak=50000)

Обучение базовых моделей...
  → linear... OOF AUC: 0.9785, Eval AUC: 0.7868
  → knn... 

## ЭТАП 3: Агрегация результатов (mean ± std)

In [ ]:
# DataFrame со всеми результатами
df_all = pd.DataFrame(all_results)
print(f"Всего результатов: {len(df_all)} (4 пресета × 2 метода × {len(SEEDS)} сидов)")

# Агрегация: mean ± std по сидам
metrics_cols = ['roc_auc', 'pr_auc', 'f1', 'precision', 'recall', 'over_refusal_rate']

agg_results = []
for (preset, method), group in df_all.groupby(['preset', 'method']):
    row = {'preset': preset, 'method': method, 'n_seeds': len(group)}
    for col in metrics_cols:
        row[f'{col}_mean'] = group[col].mean()
        row[f'{col}_std'] = group[col].std()
    agg_results.append(row)

df_agg = pd.DataFrame(agg_results)
df_agg = df_agg.sort_values(['preset', 'method']).reset_index(drop=True)

print(f"\nАгрегированные результаты ({len(SEEDS)} seeds):")
display(df_agg[['preset', 'method', 'n_seeds', 'roc_auc_mean', 'roc_auc_std', 'f1_mean', 'over_refusal_rate_mean']].round(4))

Всего результатов: 24 (4 пресета × 2 метода × 3 сидов)

Агрегированные результаты (3 seeds):


,preset,method,n_seeds,roc_auc_mean,roc_auc_std,f1_mean,over_refusal_rate_mean
0,0.2.0/classic-light,soft-voting,3,0.7989,0.0065,0.9119,0.5000
1,0.2.0/classic-light,stacking,3,0.7962,0.0076,0.9044,0.4762
2,0.2.0/classic-medium,soft-voting,3,0.8051,0.0035,0.8974,0.4254
3,0.2.0/classic-medium,stacking,3,0.7878,0.0078,0.9096,0.4825
4,0.3.0/classic-light,soft-voting,3,0.7989,0.0065,0.9119,0.5000
5,0.3.0/classic-light,stacking,3,0.7962,0.0076,0.9044,0.4762
6,0.3.0/classic-medium,soft-voting,3,0.8046,0.0037,0.9056,0.4683
7,0.3.0/classic-medium,stacking,3,0.7934,0.0077,0.9047,0.4762


## ЭТАП 4: Сводная таблица с референсами

Референсы из `metrics_summary_agg.csv` (full mode, mean по 3 seeds):

| Model | ROC-AUC | PR-AUC | F1 | ORR |
|-------|---------|--------|-----|-----|
| AutoGluon | 0.806 ± 0.014 | 0.973 | 0.875 | 0.375 |
| LightAutoML | 0.815 ± 0.002 | 0.974 | 0.887 | 0.375 |
| Linear (pipeline) | 0.789 ± 0.004 | 0.970 | 0.860 | 0.364 |
| KNN (pipeline) | 0.757 ± 0.018 | 0.959 | 0.908 | 0.621 |

In [ ]:
print("="*120)
print("СВОДНАЯ ТАБЛИЦА: Стекинг vs AutoML (mean ± std по 3 сидам)")
print("="*120)
print("\n⚠️ Порог 0.5 — стандартный. Для бизнес-задачи порог надо подбирать отдельно!")

def fmt_mean_std(mean, std):
    return f"{mean:.4f}±{std:.4f}"

print(f"\n{'Preset':<30} {'Method':<12} {'ROC-AUC':<16} {'PR-AUC':<16} {'F1':<16} {'ORR':<16}")
print("-"*120)

for _, row in df_agg.iterrows():
    roc = fmt_mean_std(row['roc_auc_mean'], row['roc_auc_std'])
    pr = fmt_mean_std(row['pr_auc_mean'], row['pr_auc_std'])
    f1 = fmt_mean_std(row['f1_mean'], row['f1_std'])
    orr = fmt_mean_std(row['over_refusal_rate_mean'], row['over_refusal_rate_std'])
    print(f"{row['preset']:<30} {row['method']:<12} {roc:<16} {pr:<16} {f1:<16} {orr:<16}")

# Референсы
print(f"\n{'─'*120}")
print("РЕФЕРЕНСЫ (из metrics_summary_agg.csv, mean ± std по 3 сидам):")
print(f"{'─'*120}")
refs = [
    ("REF: Linear (pipeline)", "-", "0.7890±0.0038", "0.9702±0.0003", "0.8599±0.0020", "0.3635±0.0099"),
    ("REF: KNN (pipeline)", "-", "0.7569±0.0179", "0.9592±0.0049", "0.9083±0.0094", "0.6206±0.0324"),
    ("REF: AutoGluon", "-", "0.8065±0.0141", "0.9733±0.0019", "0.8746±0.0221", "0.3746±0.0310"),
    ("REF: LightAutoML", "-", "0.8149±0.0020", "0.9743±0.0001", "0.8868±0.0015", "0.3746±0.0073"),
]
for ref in refs:
    print(f"{ref[0]:<30} {ref[1]:<12} {ref[2]:<16} {ref[3]:<16} {ref[4]:<16} {ref[5]:<16}")

СВОДНАЯ ТАБЛИЦА: Стекинг vs AutoML (mean ± std по 3 сидам)

⚠️ Порог 0.5 — стандартный. Для бизнес-задачи порог надо подбирать отдельно!

Preset                         Method       ROC-AUC          PR-AUC           F1               ORR             
------------------------------------------------------------------------------------------------------------------------
0.2.0/classic-light            soft-voting  0.7989±0.0065    0.9716±0.0010    0.9119±0.0015    0.5000±0.0172   
0.2.0/classic-light            stacking     0.7962±0.0076    0.9712±0.0012    0.9044±0.0014    0.4762±0.0126   
0.2.0/classic-medium           soft-voting  0.8051±0.0035    0.9729±0.0003    0.8974±0.0031    0.4254±0.0073   
0.2.0/classic-medium           stacking     0.7878±0.0078    0.9657±0.0011    0.9096±0.0034    0.4825±0.0153   
0.3.0/classic-light            soft-voting  0.7989±0.0065    0.9716±0.0010    0.9119±0.0015    0.5000±0.0172   
0.3.0/classic-light            stacking     0.7962±0.0076    0.9712±0

In [ ]:
# === КЛЮЧЕВЫЕ СРАВНЕНИЯ ===
print("\n" + "="*80)
print("КЛЮЧЕВЫЕ СРАВНЕНИЯ (усреднённые по 3 сидам)")
print("="*80)

# Лучший стекинг
stacking_only = df_agg[df_agg['method'] == 'stacking']
best_stack = stacking_only.loc[stacking_only['roc_auc_mean'].idxmax()]
best_roc_mean = best_stack['roc_auc_mean']
best_roc_std = best_stack['roc_auc_std']
best_preset = best_stack['preset']
best_orr = best_stack['over_refusal_rate_mean']

# Лучший soft-voting
soft_only = df_agg[df_agg['method'] == 'soft-voting']
best_soft = soft_only.loc[soft_only['roc_auc_mean'].idxmax()]
best_soft_roc = best_soft['roc_auc_mean']
best_soft_std = best_soft['roc_auc_std']

print(f"\n1. ЛУЧШИЙ СТЕКИНГ: {best_preset}")
print(f"   ROC-AUC: {best_roc_mean:.4f} ± {best_roc_std:.4f}")
print(f"   ORR:     {best_orr:.4f}")

print(f"\n2. ЛУЧШИЙ SOFT-VOTING: {best_soft['preset']}")
print(f"   ROC-AUC: {best_soft_roc:.4f} ± {best_soft_std:.4f}")

# 0.2.0 vs 0.3.0 (classic-medium, stacking)
medium_020 = df_agg[(df_agg['preset'] == '0.2.0/classic-medium') & (df_agg['method'] == 'stacking')]['roc_auc_mean'].values[0]
medium_030 = df_agg[(df_agg['preset'] == '0.3.0/classic-medium') & (df_agg['method'] == 'stacking')]['roc_auc_mean'].values[0]
delta_catboost = medium_020 - medium_030

print(f"\n3. 0.2.0 vs 0.3.0 (classic-medium, стекинг):")
print(f"   0.2.0 (с catboost): {medium_020:.4f}")
print(f"   0.3.0 (без catboost): {medium_030:.4f}")
print(f"   Δ (потеря от удаления catboost): {delta_catboost:+.4f}")

# Gap до AutoML
best_overall = max(best_roc_mean, best_soft_roc)
gap_autogluon = 0.8065 - best_overall
gap_lama = 0.8149 - best_overall

print(f"\n4. GAP до внешних AutoML (от лучшего {best_overall:.4f}):")
print(f"   До AutoGluon (0.8065): {gap_autogluon:+.4f}")
print(f"   До LightAutoML (0.8149): {gap_lama:+.4f}")

# ORR сравнение
print(f"\n5. ORR стекинга vs одиночных:")
print(f"   Лучший стекинг: {best_orr:.4f}")
print(f"   Linear (pipeline): 0.364")
print(f"   KNN (pipeline): 0.621")
print(f"   AutoGluon/LAMA: 0.375")


КЛЮЧЕВЫЕ СРАВНЕНИЯ (усреднённые по 3 сидам)

1. ЛУЧШИЙ СТЕКИНГ: 0.2.0/classic-light
   ROC-AUC: 0.7962 ± 0.0076
   ORR:     0.4762

2. ЛУЧШИЙ SOFT-VOTING: 0.2.0/classic-medium
   ROC-AUC: 0.8051 ± 0.0035

3. 0.2.0 vs 0.3.0 (classic-medium, стекинг):
   0.2.0 (с catboost): 0.7878
   0.3.0 (без catboost): 0.7934
   Δ (потеря от удаления catboost): -0.0057

4. GAP до внешних AutoML (от лучшего 0.8051):
   До AutoGluon (0.8065): +0.0014
   До LightAutoML (0.8149): +0.0098

5. ORR стекинга vs одиночных:
   Лучший стекинг: 0.4762
   Linear (pipeline): 0.364
   KNN (pipeline): 0.621
   AutoGluon/LAMA: 0.375


## ЭТАП 5: ROC-AUC кривые

Визуализация ROC-кривых для всех сравниваемых прогонов (seed=42 как репрезентативный).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

# === ROC-AUC КРИВЫЕ ===
# Используем seed=42 как репрезентативный для визуализации
PLOT_SEED = 42

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Левый график: базовые модели + ансамбли (0.2.0/classic-medium) ---
ax1 = axes[0]
ax1.set_title(f"ROC-кривые: Базовые модели vs Ансамбли (seed={PLOT_SEED})", fontsize=12)

# Базовые модели
colors_base = {"linear": "blue", "knn": "green", "randomforest": "orange", "catboost": "red"}
for model_name in ["linear", "knn", "randomforest", "catboost"]:
    key = (PLOT_SEED, f"base:{model_name}", "single")
    if key in all_predictions:
        y_prob = all_predictions[key]
        fpr, tpr, _ = roc_curve(eval_labels, y_prob)
        auc_val = roc_auc_score(eval_labels, y_prob)
        ax1.plot(fpr, tpr, label=f"{model_name} (AUC={auc_val:.4f})", 
                 color=colors_base[model_name], linestyle="--", alpha=0.7)

# Ансамбли для 0.2.0/classic-medium
preset = "0.2.0/classic-medium"
for method, ls, lw in [("soft-voting", "-", 2.5), ("stacking", "-.", 2.5)]:
    key = (PLOT_SEED, preset, method)
    if key in all_predictions:
        y_prob = all_predictions[key]
        fpr, tpr, _ = roc_curve(eval_labels, y_prob)
        auc_val = roc_auc_score(eval_labels, y_prob)
        ax1.plot(fpr, tpr, label=f"{method} (AUC={auc_val:.4f})", 
                 linestyle=ls, linewidth=lw, color="purple" if method == "soft-voting" else "brown")

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3, label="Random (AUC=0.5)")
ax1.set_xlabel("False Positive Rate (1 - Specificity)")
ax1.set_ylabel("True Positive Rate (Sensitivity)")
ax1.legend(loc="lower right", fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Правый график: все ансамбли (4 пресета × 2 метода) ---
ax2 = axes[1]
ax2.set_title(f"ROC-кривые: Все ансамбли (seed={PLOT_SEED})", fontsize=12)

colors_preset = {
    "0.2.0/classic-light": "blue",
    "0.2.0/classic-medium": "purple",
    "0.3.0/classic-light": "cyan",
    "0.3.0/classic-medium": "magenta",
}

for preset in PRESETS.keys():
    for method, ls in [("soft-voting", "-"), ("stacking", "--")]:
        key = (PLOT_SEED, preset, method)
        if key in all_predictions:
            y_prob = all_predictions[key]
            fpr, tpr, _ = roc_curve(eval_labels, y_prob)
            auc_val = roc_auc_score(eval_labels, y_prob)
            label = f"{preset.split('/')[1]} {method[:5]} ({auc_val:.4f})"
            ax2.plot(fpr, tpr, label=label, color=colors_preset[preset], 
                     linestyle=ls, linewidth=1.8, alpha=0.8)

# Референсные линии для AutoML (горизонтальные пунктиры на уровне их AUC)
ax2.axhline(y=0.806, color='gray', linestyle=':', alpha=0.5, label='AutoGluon ref (0.806)')
ax2.axhline(y=0.815, color='gray', linestyle=':', alpha=0.7, label='LightAutoML ref (0.815)')

ax2.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax2.set_xlabel("False Positive Rate (1 - Specificity)")
ax2.set_ylabel("True Positive Rate (Sensitivity)")
ax2.legend(loc="lower right", fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curves_stacking.png", dpi=150, bbox_inches='tight')
print(f"📊 Сохранено: {OUTPUT_DIR / 'roc_curves_stacking.png'}")
plt.show()

In [ ]:
# === ВЕРДИКТ ===
print("\n" + "="*80)
print("ВЕРДИКТ")
print("="*80)

best_overall = max(best_roc_mean, best_soft_roc)

if best_overall >= 0.806:
    scenario = "✅ СЦЕНАРИЙ 1: Стекинг ≈ AutoML"
    print(f"\n{scenario}")
    print(f"   Лучший результат {best_overall:.4f} сравним с AutoGluon (0.806).")
    print(f"   → Разрыв закрывается ансамблем на тех же e5-фичах.")
    print(f"   → Узкое место было в ОТСУТСТВИИ стекинга, не в эмбеддере.")
elif best_overall >= 0.79:
    scenario = "⚠️ СЦЕНАРИЙ 2: Стекинг частично помогает"
    print(f"\n{scenario}")
    print(f"   Лучший результат {best_overall:.4f} (между linear 0.789 и AutoML 0.806).")
    print(f"   → Ансамбль помогает частично.")
    print(f"   → Остаток разрыва за чем-то ещё (калибровка, внутренние трюки AutoML).")
else:
    scenario = "❌ СЦЕНАРИЙ 3: Ансамбль не помогает"
    print(f"\n{scenario}")
    print(f"   Лучший результат {best_overall:.4f} (≤ linear 0.789).")
    print(f"   → Потолок ~0.79 в представлении.")
    print(f"   → Бюджет в эмбеддинг (fine-tune, другой эмбеддер).")

# Вердикт по CatBoost
print(f"\n{'─'*60}")
print(f"ВЕРДИКТ по CatBoost (0.2.0 vs 0.3.0):")
if delta_catboost > 0.01:
    print(f"   Удаление catboost стоило {delta_catboost:+.4f} ROC-AUC.")
    print(f"   → СИЛЬНЫЙ аргумент ВЕРНУТЬ GBM.")
elif delta_catboost > 0.005:
    print(f"   Удаление catboost стоило {delta_catboost:+.4f} — умеренный эффект.")
elif delta_catboost > 0:
    print(f"   Удаление catboost стоило {delta_catboost:+.4f} — незначительно.")
else:
    print(f"   Удаление catboost НЕ ухудшило результат ({delta_catboost:+.4f}).")
    print(f"   → Правильное решение убрать.")


ВЕРДИКТ

⚠️ СЦЕНАРИЙ 2: Стекинг частично помогает
   Лучший результат 0.8051 (между linear 0.789 и AutoML 0.806).
   → Ансамбль помогает частично.
   → Остаток разрыва за чем-то ещё (калибровка, внутренние трюки AutoML).

────────────────────────────────────────────────────────────
ВЕРДИКТ по CatBoost (0.2.0 vs 0.3.0):
   Удаление catboost НЕ ухудшило результат (-0.0057).
   → Правильное решение убрать.


In [ ]:
# === РЕЗЮМЕ ===
print("\n" + "="*80)
print("РЕЗЮМЕ")
print("="*80)

print(f"\n📋 4 НАБОРА СКОРЕРОВ (верифицировано из YAML):")
print(f"   0.2.0/classic-light:  {scorers_020_light}")
print(f"   0.2.0/classic-medium: {scorers_020_medium}")
print(f"   0.3.0/classic-light:  {scorers_030_light}")
print(f"   0.3.0/classic-medium: {scorers_030_medium} (БЕЗ catboost!)")

print(f"\n📊 ТАБЛИЦА СТЕКИНГА (ROC-AUC, mean ± std по {len(SEEDS)} сидам):")
for _, row in df_agg.iterrows():
    print(f"   {row['preset']:<25} {row['method']:<12}: {row['roc_auc_mean']:.4f} ± {row['roc_auc_std']:.4f}")

print(f"\n🎯 РЕАЛИЗОВАВШИЙСЯ ИСХОД:")
print(f"   Лучший результат: {best_overall:.4f}")
print(f"   Gap до AutoGluon (0.806): {gap_autogluon:+.4f}")
print(f"   Gap до LightAutoML (0.815): {gap_lama:+.4f}")

print(f"\n⚖️ ВЕРДИКТ по CatBoost:")
print(f"   Δ (0.2.0 vs 0.3.0 medium stacking): {delta_catboost:+.4f} ROC-AUC")

print(f"\n📁 ПУТИ К ДАННЫМ:")
print(f"   100k эмбеддинги: {EMB_CACHE}/intfloat_multilingual-e5-large-instruct_full100k_seed{{42,123,456}}.npy")
print(f"   100k JSON: {DATA_DIR}/wildjailbreak_full100k_seed{{42,123,456}}.json")
print(f"   Eval: {EVAL_EMB} ({len(eval_labels)} строк)")


РЕЗЮМЕ

📋 4 НАБОРА СКОРЕРОВ (верифицировано из YAML):
   0.2.0/classic-light:  ['knn', 'linear', 'mlknn']
   0.2.0/classic-medium: ['knn', 'linear', 'mlknn', 'catboost', 'sklearn:RandomForestClassifier']
   0.3.0/classic-light:  ['knn', 'linear', 'mlknn']
   0.3.0/classic-medium: ['knn', 'linear', 'mlknn', 'sklearn:RandomForestClassifier'] (БЕЗ catboost!)

📊 ТАБЛИЦА СТЕКИНГА (ROC-AUC, mean ± std по 3 сидам):
   0.2.0/classic-light       soft-voting : 0.7989 ± 0.0065
   0.2.0/classic-light       stacking    : 0.7962 ± 0.0076
   0.2.0/classic-medium      soft-voting : 0.8051 ± 0.0035
   0.2.0/classic-medium      stacking    : 0.7878 ± 0.0078
   0.3.0/classic-light       soft-voting : 0.7989 ± 0.0065
   0.3.0/classic-light       stacking    : 0.7962 ± 0.0076
   0.3.0/classic-medium      soft-voting : 0.8046 ± 0.0037
   0.3.0/classic-medium      stacking    : 0.7934 ± 0.0077

🎯 РЕАЛИЗОВАВШИЙСЯ ИСХОД:
   Лучший результат: 0.8051
   Gap до AutoGluon (0.806): +0.0014
   Gap до LightAutoML (0

In [ ]:
# Сохраняем результаты
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Все результаты по сидам
df_all.to_csv(OUTPUT_DIR / "stacking_scorers_full100k_all_seeds.csv", index=False)
print(f"Сохранено (все сиды): {OUTPUT_DIR / 'stacking_scorers_full100k_all_seeds.csv'}")

# Агрегированные результаты
df_agg.to_csv(OUTPUT_DIR / "stacking_scorers_full100k_agg.csv", index=False)
print(f"Сохранено (агрегация): {OUTPUT_DIR / 'stacking_scorers_full100k_agg.csv'}")

Сохранено (все сиды): /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/results/diagnostics/stacking_scorers_full100k_all_seeds.csv
Сохранено (агрегация): /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/results/diagnostics/stacking_scorers_full100k_agg.csv


---

## ПРИЛОЖЕНИЕ: Справочный результат на adversarial-only train

**⚠️ НЕ для сравнения с AutoML!** AutoML обучались на полном 100k, этот эксперимент — на 6k adversarial-only.

In [ ]:
print("="*80)
print("СПРАВОЧНО: Adversarial-only train (6k)")
print("⚠️ НЕ для сравнения с AutoML — разный train!")
print("="*80)

adv_results = {
    "0.2.0/classic-medium stacking (adv-only)": 0.792,
    "CatBoost одиночный (adv-only, ноутбук 09)": 0.784,
}

print(f"\nРезультаты на adversarial-only train (6k):")
for name, roc in adv_results.items():
    print(f"   {name}: ROC-AUC {roc:.4f}")

print(f"\nИнтерпретация:")
print(f"   Adversarial-only — это 'потолок на трудном подмножестве'.")
print(f"   Полный train даёт больше сигнала, поэтому метрики выше.")

СПРАВОЧНО: Adversarial-only train (6k)
⚠️ НЕ для сравнения с AutoML — разный train!

Результаты на adversarial-only train (6k):
   0.2.0/classic-medium stacking (adv-only): ROC-AUC 0.7920
   CatBoost одиночный (adv-only, ноутбук 09): ROC-AUC 0.7840

Интерпретация:
   Adversarial-only — это 'потолок на трудном подмножестве'.
   Полный train даёт больше сигнала, поэтому метрики выше.
